# EcoEye — Bird Image Scraper v3: Species Names + Datetime Filenames

**Iteration:** Adds species name prefix to each downloaded filename alongside a datetime timestamp.  
**Source:** iStock paginated image search  
**Improvement over v2:** Images are now named `Albatross_DD-MM-YYYY-HH-MM-SS.jpg` instead of generic timestamps, making the dataset self-documenting.

> ✅ This is the version used to build the labelled training dataset for the ML model.
> See `Final Scraping (Code).ipynb` for the fully refactored production version.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import requests
from datetime import datetime
import os
import time

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────────────────────────
SPECIES_NAME      = "Albatross"       # Species label — used in filename and folder name
CHROMEDRIVER_PATH = "chromedriver.exe" # Place chromedriver.exe in this folder, or provide full path
START_PAGE        = 1
END_PAGE          = 29

SAVE_FOLDER = os.path.join(os.getcwd(), "downloaded_images", SPECIES_NAME)
os.makedirs(SAVE_FOLDER, exist_ok=True)
print(f"Saving images to: {SAVE_FOLDER}")

In [ ]:
def generate_image_name(species):
    """Generate a unique filename: <species>_<timestamp>.jpg"""
    return f"{species}_" + datetime.now().strftime("%d-%m-%Y-%H-%M-%S-%f") + ".jpg"


def setup_driver(driver_path):
    """Initialize and return a Chrome WebDriver instance."""
    service = Service(driver_path)
    return webdriver.Chrome(service=service)


def scrape_images(driver, url):
    """Navigate to a URL and return all image src URLs found on the page."""
    driver.get(url)
    time.sleep(5)  # Wait for dynamic content to load
    image_elements = driver.find_elements(By.TAG_NAME, 'img')
    return [
        img.get_attribute('src')
        for img in image_elements
        if img.get_attribute('src')
    ]


def download_images(image_urls, save_folder, species):
    """Download and save a list of image URLs to the save_folder."""
    downloaded = 0
    for image_url in image_urls:
        response = requests.get(image_url)
        if response.status_code == 200:
            image_name = generate_image_name(species)
            save_path = os.path.join(save_folder, image_name)
            with open(save_path, 'wb') as f:
                f.write(response.content)
            downloaded += 1
            print(f"  Saved: {image_name}")
        else:
            print(f"  Failed ({response.status_code}): {image_url}")
    return downloaded


def main(species, base_url, start_page, end_page, driver_path, save_folder):
    """Scrape images across paginated results and save them locally."""
    driver = setup_driver(driver_path)
    total = 0
    try:
        for page in range(start_page, end_page + 1):
            url = base_url.format(page=page)
            print(f"\nPage {page}: {url}")
            images = scrape_images(driver, url)
            count = download_images(images, save_folder, species)
            total += count
            print(f"  → {count} images downloaded")
    finally:
        driver.quit()
    print(f"\nDone. Total: {total} images → {save_folder}")

In [ ]:
# ─── RUN ───────────────────────────────────────────────────────────────────────
BASE_URL = "https://www.istockphoto.com/search/2/image?mediatype=photography&page={page}&phrase=" + SPECIES_NAME.lower()

main(
    species=SPECIES_NAME,
    base_url=BASE_URL,
    start_page=START_PAGE,
    end_page=END_PAGE,
    driver_path=CHROMEDRIVER_PATH,
    save_folder=SAVE_FOLDER
)